In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/SnowPole_Detection_Dataset/
# !git clone https://github.com/ultralytics/ultralytics.git

/content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset


In [ ]:
!pip uninstall -y ultralytics

In [ ]:
!rm -rf /content/ultralytics
!git clone https://github.com/MuhammadIbneRafiq/ultralytics4channel /content/ultralytics

Cloning into '/content/ultralytics'...
remote: Enumerating objects: 276, done.
remote: Counting objects: 100% (276/276), done.
remote: Compressing objects: 100% (222/222), done.
remote: Total 276 (delta 65), reused 254 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (276/276), 706.61 KiB | 1.57 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [ ]:
!pip install -q ultralytics==8.2.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 53.9 MB/s eta 0:00:00


In [ ]:
import sys
sys.path.insert(0, "/content")  # parent of the ultralytics package

import ultralytics
from ultralytics import YOLO

print("Ultralytics module file:", ultralytics.__file__)


Ultralytics module file: /content/ultralytics/__init__.py


In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path
import shutil
import cv2
import numpy as np
import yaml
from tqdm import tqdm

from ultralytics import YOLO
import torch

from torch.utils.data import Dataset, DataLoader

In [ ]:
COMB_ROOT  = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/images")

# 1-channel range-normalized images
RANGE_ROOT = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/range-normalized-continuous")

# New 4-channel dual-input dataset
DUAL_ROOT  = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range")
DUAL_ROOT.mkdir(parents=True, exist_ok=True)

print("COMB_ROOT :", COMB_ROOT)
print("RANGE_ROOT:", RANGE_ROOT)
print("DUAL_ROOT :", DUAL_ROOT)

COMB_ROOT : /content/drive/MyDrive/SnowPole_Detection_Dataset/images
RANGE_ROOT: /content/drive/MyDrive/SnowPole_Detection_Dataset/range-normalized-continuous
DUAL_ROOT : /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range


In [ ]:
# !yolo train model=yolov9t.pt epochs=150 imgsz=1024 device=0 batch=2 data=/content/drive/MyDrive/data.yaml project=/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec

In [ ]:
# !yolo val \
#   model=/content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec-v11n/train2/weights/best.pt \
#   data=/content/drive/MyDrive/data.yaml \
#   split=test \
#   imgsz=1024 \
#   device=0 \
#   batch=16 \
#   project="comb5_signal_reflec_range_11n" \
#   name="comb5_signal_reflec_range_11n_test_eval"


In [ ]:
def make_dual_split_npy(split: str,
                        comb_root: Path,
                        range_root: Path,
                        dual_root: Path,
                        save_as_png: bool = True):
    """
    Create 4-channel dual images by stacking comb RGB (BGR) + range (.npy float32 [0,1]).
    - comb_root: root containing comb_root/images/<split>/*.png and comb_root/labels/<split>/*.txt
    - range_root: root containing range_root/<split>/*.npy (each named like the comb image stem)
    - dual_root: destination root; will create dual_root/images/<split> and dual_root/labels/<split>
    - save_as_png: if True, save stacked RGBA PNGs (4 channel) so existing YOLO loaders can read them.
                   (range channel is quantized to uint8 for the PNG; the original .npy is left unchanged)
    """
    comb_img_dir   = comb_root / split
    src_lbl_dir    = comb_root / "../" / "labels" / split
    range_npy_dir  = range_root / split
    dual_img_dir   = dual_root / "images" / split
    dual_lbl_dir   = dual_root / "labels" / split

    # if this split already has images, skip doing anything
    if dual_img_dir.exists() and any(dual_img_dir.glob("*.png")):
        print(f"[{split}] dual images already exist in {dual_img_dir}, skipping.")
        return

    dual_img_dir.mkdir(parents=True, exist_ok=True)
    dual_lbl_dir.mkdir(parents=True, exist_ok=True)

    # copy labels from comb labels to dual labels (they are the same)
    if src_lbl_dir.exists():
        for lbl in src_lbl_dir.glob("*.txt"):
            # copy2 preserves timestamps, etc.
            shutil.copy2(lbl, dual_lbl_dir / lbl.name)
    else:
        print(f"Warning: source label dir not found: {src_lbl_dir}")

    # gather comb images
    img_files = sorted(comb_img_dir.glob("*.*"))
    print(f"[{split}] comb images found: {len(img_files)}")

    for comb_path in tqdm(img_files, desc=f"make_dual_split ({split})"):
        stem = comb_path.stem

        # read comb RGB (OpenCV: BGR)
        comb = cv2.imread(str(comb_path), cv2.IMREAD_COLOR)
        if comb is None:
            print("Could not read comb image:", comb_path)
            continue

        # read corresponding range .npy
        range_npy_path = range_npy_dir / f"{stem}.npy"
        if not range_npy_path.exists():
            # try alternative stem patterns if needed
            print(f"Missing range .npy for {stem} -> {range_npy_path} (skipping)")
            continue

        try:
            range_arr = np.load(str(range_npy_path))   # expected float32 in [0,1]
        except Exception as e:
            print(f"Failed to load {range_npy_path}: {e}")
            continue

        # squeeze any extra dims
        if range_arr.ndim == 3 and range_arr.shape[0] in (1,):
            range_arr = np.squeeze(range_arr, axis=0)
        if range_arr.ndim != 2:
            # if it has a channel dim (H,W,1) -> squeeze
            if range_arr.ndim == 3 and range_arr.shape[2] == 1:
                range_arr = np.squeeze(range_arr, axis=2)
            else:
                print(f"Unexpected shape for range npy {range_npy_path}: {range_arr.shape} (skipping)")
                continue

        # ensure float32 and clip to [0,1]
        range_arr = range_arr.astype(np.float32)
        range_arr = np.clip(range_arr, 0.0, 1.0)

        # convert range to uint8 for stacking if saving PNGs (visualization/training with standard loader)
        range_uint8 = (range_arr * 255.0).astype(np.uint8)

        # resize range to comb dims if necessary (note cv2 resize expects (width, height))
        if range_uint8.shape != comb.shape[:2]:
            range_uint8 = cv2.resize(range_uint8, (comb.shape[1], comb.shape[0]), interpolation=cv2.INTER_NEAREST)

        # stack into 4-channel: B, G, R, RANGE
        rgba = np.dstack([comb, range_uint8])  # result dtype uint8, shape (H, W, 4)

        # write stacked 4-channel PNG so YOLO-like image loaders can ingest it
        if save_as_png:
            out_img_path = dual_img_dir / f"{stem}.png"
            # OpenCV will write all 4 channels to PNG when given a 4-channel array.
            cv2.imwrite(str(out_img_path), rgba)

    print(f"[{split}] done. Dual images written to {dual_img_dir}, labels copied to {dual_lbl_dir}")


# Run for all splits (call this cell)
for split in ["train", "valid", "test"]:
    make_dual_split_npy(split, comb_root=COMB_ROOT, range_root=RANGE_ROOT, dual_root=DUAL_ROOT, save_as_png=True)


[train] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/train, skipping.
[valid] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/valid, skipping.
[test] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/test, skipping.


In [ ]:
ORIG_DATA_YAML = COMB_ROOT / "../" /"data.yaml"
DUAL_DATA_YAML = DUAL_ROOT / "data.yaml"

# with open(ORIG_DATA_YAML, "r") as f:
#     cfg = yaml.safe_load(f)

# base = DUAL_ROOT

# def make_rel(p):
#     # p might be absolute or relative – we point to new dual root
#     p = Path(p)
#     return str((base / "images" / p.name).parent)  # keep split names

# # If your original yaml used explicit paths, you can instead do:
# # cfg["path"]  = str(DUAL_ROOT)
# cfg["path"]  = str(DUAL_ROOT)
# cfg["train"] = "images/train"
# cfg["valid"]   = "images/valid"
# cfg["test"]  = "images/test"
# cfg["channels"] = 4          # tell YOLO this is 4-channel data with the RGB-Alpha

# with open(DUAL_DATA_YAML, "w") as f:
#     yaml.safe_dump(cfg, f)

# print(DUAL_DATA_YAML.read_text())

# Enhanced Loss Functions for Dual Network Training

In this notebook, we'll implement and compare three specialized loss functions designed to improve the performance of dual-network architectures that process both RGB and range/depth data:

1. **Depth-Aware Loss**: Weighs detection errors based on depth information, focusing on objects at certain depth ranges while reducing penalties for errors in challenging depth regions.

2. **Cross-Modal Attention Loss**: Encourages the network to focus attention on regions where RGB and range data provide complementary information, improving feature fusion.

3. **Feature Consistency Loss**: Ensures consistent feature representations between RGB and range modalities for the same objects, while preserving modality-specific details.

We'll compare these loss functions against a baseline model using the standard YOLO detection loss.

In [ ]:
from ultralytics import YOLO
import torch
from ultralytics.nn.tasks import DetectionModel

import torch
from ultralytics.nn.tasks import DetectionModel
# from ultralytics.nn.modules.block import ELAN1, AConv, RepNCSPELAN4, SPPELAN

# allow the DetectionModel class for unpickling

torch.serialization.add_safe_globals([
    DetectionModel,
    # ELAN1,
    # AConv,
    # RepNCSPELAN4,
    # SPPELAN,
])

# Base model for standard training (without custom loss)
model = YOLO("yolov8n.pt")  # or YOLO("yolov9t.pt", task="detect")
model.model.eval()

# Standard training with default loss function
model.train(
    data=str(DUAL_DATA_YAML),
    epochs=100,  # Reduced from 400 to make experiments faster
    imgsz=1024,
    device=0,
    amp=False,  # Disable AMP
    batch=16,
    project="dual_comb_range_experiments",
    name="standard_baseline",
)

In [ ]:
# Create a utils directory in the notebook folder if it doesn't exist
import os
os.makedirs("utils", exist_ok=True)

# Define custom loss functions
%%writefile utils/dual_loss.py
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

"""
Dual network loss functions for RGB+Range detection.
This module implements specialized loss functions for multi-modal inputs combining RGB and range/depth information.
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils.metrics import bbox_iou


class DepthAwareLoss:
    """
    Depth-Aware Loss for dual network detection with RGB+Range data.
    
    This loss function weighs detection errors based on depth/range information,
    giving higher importance to foreground objects at certain depth ranges and 
    reducing the penalty for errors in challenging depth regions.
    """
    
    def __init__(self, depth_weight=0.5, depth_focus_range=(0.2, 0.8), depth_sigma=0.3):
        """
        Initialize DepthAwareLoss with configurable parameters.
        
        Args:
            depth_weight (float): Weight for depth-aware term in total loss
            depth_focus_range (tuple): Range of normalized depth values to focus on (min, max)
            depth_sigma (float): Sigma for Gaussian weighting of depth values
        """
        self.depth_weight = depth_weight
        self.focus_min, self.focus_max = depth_focus_range
        self.depth_sigma = depth_sigma
    
    def __call__(self, pred_bboxes, target_bboxes, depth_map):
        """
        Calculate depth-aware loss component.
        
        Args:
            pred_bboxes (torch.Tensor): Predicted bounding boxes (N, 4) in xywh format
            target_bboxes (torch.Tensor): Target bounding boxes (N, 4) in xywh format
            depth_map (torch.Tensor): Depth/range map of the input image
            
        Returns:
            torch.Tensor: Depth-weighted loss value
        """
        if len(pred_bboxes) == 0 or len(target_bboxes) == 0:
            return torch.tensor(0.0, device=pred_bboxes.device)
            
        # Calculate standard bounding box loss (IoU-based)
        iou_loss = 1.0 - bbox_iou(pred_bboxes, target_bboxes, xywh=True, GIoU=True)
        
        # Extract depth values at target center points
        target_centers = target_bboxes[:, :2]  # Get xy center points
        h, w = depth_map.shape[-2:]
        
        # Scale center points to depth map coordinates
        target_centers_scaled = target_centers.clone()
        target_centers_scaled[:, 0] *= w
        target_centers_scaled[:, 1] *= h
        target_centers_scaled = target_centers_scaled.long().clamp(0, w-1, 0, h-1)
        
        # Extract depth values at target centers
        depth_values = torch.zeros(len(target_centers), device=target_bboxes.device)
        for i, (x, y) in enumerate(target_centers_scaled):
            depth_values[i] = depth_map[0, 0, y, x]  # Assuming depth_map has shape [B, 1, H, W]
        
        # Calculate depth-based weights using Gaussian distribution
        # Focus weight is highest within focus_range and drops off outside
        focus_weights = torch.exp(-torch.pow(depth_values - (self.focus_min + self.focus_max)/2, 2) / 
                                 (2 * self.depth_sigma**2))
                                 
        # Apply depth-based weights to IoU loss
        weighted_loss = iou_loss * focus_weights
        return weighted_loss.mean() * self.depth_weight


class CrossModalAttentionLoss:
    """
    Cross-Modal Attention Loss for dual network detection.
    
    This loss function encourages the network to focus attention on regions
    where RGB and range data provide complementary information, improving
    feature fusion and multi-modal reasoning.
    """
    
    def __init__(self, attention_weight=0.3):
        """
        Initialize CrossModalAttentionLoss.
        
        Args:
            attention_weight (float): Weight for attention loss in total loss
        """
        self.attention_weight = attention_weight
        self.mse = nn.MSELoss(reduction='mean')
    
    def __call__(self, rgb_features, range_features, fg_mask=None):
        """
        Calculate cross-modal attention loss.
        
        Args:
            rgb_features (torch.Tensor): Features from RGB branch
            range_features (torch.Tensor): Features from range/depth branch
            fg_mask (torch.Tensor, optional): Foreground mask to focus attention
            
        Returns:
            torch.Tensor: Cross-modal attention loss
        """
        # Calculate channel-wise attention weights
        rgb_attention = self._get_attention_map(rgb_features)
        range_attention = self._get_attention_map(range_features)
        
        # If foreground mask is provided, focus attention on foreground areas
        if fg_mask is not None:
            # Resize mask to match attention maps
            fg_mask = F.interpolate(fg_mask.float(), size=rgb_attention.shape[-2:], 
                                   mode='nearest').bool()
            
            # Apply mask
            rgb_attention = rgb_attention * fg_mask
            range_attention = range_attention * fg_mask
            
        # Compute Jensen-Shannon Divergence between attention maps
        # This encourages attention to be focused on complementary regions
        # while maintaining some similarity in important areas
        attention_similarity = self._jensen_shannon_div(rgb_attention, range_attention)
        
        # We want some similarity but not perfect alignment (encouraging complementary focus)
        # So we aim for a moderate similarity value
        target_similarity = torch.ones_like(attention_similarity) * 0.5
        
        # MSE loss pushes similarity toward 0.5 (balanced attention)
        attention_loss = self.mse(attention_similarity, target_similarity)
        
        return attention_loss * self.attention_weight
        
    def _get_attention_map(self, features):
        """Generate channel attention map from features."""
        # Global average pooling along channels
        channel_avg = torch.mean(features, dim=1, keepdim=True)
        # Apply softmax to get attention probabilities
        return torch.sigmoid(channel_avg)
        
    def _jensen_shannon_div(self, p, q):
        """Calculate Jensen-Shannon divergence between two distributions."""
        # Clamp for numerical stability
        p = torch.clamp(p, min=1e-7, max=1.0)
        q = torch.clamp(q, min=1e-7, max=1.0)
        
        # Mixture distribution
        m = 0.5 * (p + q)
        
        # KL divergence terms
        kl_p_m = p * (torch.log(p) - torch.log(m))
        kl_q_m = q * (torch.log(q) - torch.log(m))
        
        # Jensen-Shannon divergence
        js_div = 0.5 * (kl_p_m.sum() + kl_q_m.sum())
        
        return js_div


class FeatureConsistencyLoss:
    """
    Feature Consistency Loss for dual network detection.
    
    This loss encourages consistent feature representations between RGB and range modalities
    for the same object, while allowing for modality-specific details to be preserved.
    """
    
    def __init__(self, consistency_weight=0.2, feature_dim=256):
        """
        Initialize FeatureConsistencyLoss.
        
        Args:
            consistency_weight (float): Weight for feature consistency loss
            feature_dim (int): Dimension for feature projection
        """
        self.consistency_weight = consistency_weight
        self.feature_dim = feature_dim
        
        # Projection layers to map features to common space
        self.rgb_proj = nn.Conv2d(feature_dim, feature_dim, kernel_size=1)
        self.range_proj = nn.Conv2d(feature_dim, feature_dim, kernel_size=1)
        
    def to(self, device):
        """Move projection layers to specific device."""
        self.rgb_proj = self.rgb_proj.to(device)
        self.range_proj = self.range_proj.to(device)
        return self
        
    def __call__(self, rgb_features, range_features, target_bboxes=None):
        """
        Calculate feature consistency loss.
        
        Args:
            rgb_features (torch.Tensor): Features from RGB branch
            range_features (torch.Tensor): Features from range branch
            target_bboxes (torch.Tensor, optional): Target bounding boxes to focus on object areas
            
        Returns:
            torch.Tensor: Feature consistency loss
        """
        # Project features to common embedding space
        rgb_proj = self.rgb_proj(rgb_features)
        range_proj = self.range_proj(range_features)
        
        if target_bboxes is not None and len(target_bboxes) > 0:
            # Extract features from object regions only
            # This focuses consistency on objects of interest
            mask = self._create_box_mask(target_bboxes, rgb_features.shape)
            rgb_proj = rgb_proj * mask
            range_proj = range_proj * mask
            
        # Normalize features for cosine similarity
        rgb_norm = F.normalize(rgb_proj, p=2, dim=1)
        range_norm = F.normalize(range_proj, p=2, dim=1)
        
        # Compute cosine similarity
        similarity = F.cosine_similarity(rgb_norm, range_norm, dim=1)
        
        # Maximize similarity (minimize negative similarity)
        consistency_loss = -similarity.mean()
        
        return consistency_loss * self.consistency_weight
        
    def _create_box_mask(self, boxes, feature_shape):
        """Create a binary mask for target boxes."""
        b, c, h, w = feature_shape
        mask = torch.zeros((b, 1, h, w), device=boxes.device)
        
        # Scale boxes to feature map size
        scaled_boxes = boxes.clone()
        scaled_boxes[:, [0, 2]] *= w  # scale x coordinates
        scaled_boxes[:, [1, 3]] *= h  # scale y coordinates
        
        # Convert to xyxy format
        x1 = (scaled_boxes[:, 0] - scaled_boxes[:, 2] / 2).long().clamp(0, w-1)
        y1 = (scaled_boxes[:, 1] - scaled_boxes[:, 3] / 2).long().clamp(0, h-1)
        x2 = (scaled_boxes[:, 0] + scaled_boxes[:, 2] / 2).long().clamp(0, w-1)
        y2 = (scaled_boxes[:, 1] + scaled_boxes[:, 3] / 2).long().clamp(0, h-1)
        
        # Fill mask with ones in box regions
        for i in range(len(boxes)):
            mask[0, 0, y1[i]:y2[i]+1, x1[i]:x2[i]+1] = 1.0
            
        return mask


class DualDetectionLoss(v8DetectionLoss):
    """
    Enhanced YOLO detection loss for dual RGB+Range networks.
    
    This class extends the standard v8DetectionLoss with additional loss terms
    specialized for dual-modality inputs, combining RGB and depth/range information.
    """
    
    def __init__(self, model):
        """
        Initialize DualDetectionLoss with model and specialized loss components.
        
        Args:
            model (DetectionModel): The YOLO detection model
        """
        super().__init__(model)
        
        # Initialize specialized dual loss components
        self.depth_aware_loss = DepthAwareLoss(depth_weight=0.5)
        self.cross_modal_loss = CrossModalAttentionLoss(attention_weight=0.3)
        self.feature_consistency_loss = FeatureConsistencyLoss(consistency_weight=0.2)
        
        # Add depth loss weight to hyperparameters
        self.hyp.depth = getattr(self.hyp, 'depth', 0.5)
        self.hyp.modal = getattr(self.hyp, 'modal', 0.3)
        self.hyp.consist = getattr(self.hyp, 'consist', 0.2)
    
    def __call__(self, preds, batch):
        """
        Calculate the combined loss for the dual detection network.
        
        Args:
            preds (Any): Model predictions
            batch (dict[str, torch.Tensor]): Batch data including images and labels
            
        Returns:
            torch.Tensor: Combined loss
        """
        # Get standard detection loss from parent class
        loss, loss_items = super().__call__(preds, batch)
        
        # Extract RGB and range features (assuming they're stored in preds)
        if isinstance(preds, tuple) and len(preds) > 2:
            rgb_features = preds[2].get('rgb_features', None)
            range_features = preds[2].get('range_features', None)
            depth_map = preds[2].get('depth_map', None)
            
            # Get batch info
            device = batch['img'].device
            dual_loss = torch.zeros(3, device=device)  # [depth, modal, consist]
            
            # Apply specialized losses if features are available
            if rgb_features is not None and range_features is not None:
                # Cross-modal attention loss
                dual_loss[1] = self.cross_modal_loss(rgb_features, range_features)
                
                # Feature consistency loss
                dual_loss[2] = self.feature_consistency_loss(rgb_features, range_features)
            
            # Depth-aware loss if depth map is available
            if depth_map is not None:
                # Extract predicted and target boxes
                pred_bboxes = preds[0]  # assuming preds[0] contains bboxes
                target_bboxes = batch['bboxes']
                
                # Calculate depth-aware loss
                dual_loss[0] = self.depth_aware_loss(pred_bboxes, target_bboxes, depth_map)
            
            # Apply loss weights
            dual_loss[0] *= self.hyp.depth
            dual_loss[1] *= self.hyp.modal
            dual_loss[2] *= self.hyp.consist
            
            # Add dual losses to standard loss
            batch_size = batch['img'].shape[0]
            dual_loss_sum = dual_loss.sum() * batch_size
            loss += dual_loss_sum
            
            # Append dual losses to loss_items for logging
            dual_loss_items = torch.cat([loss_items, dual_loss])
            return loss, dual_loss_items
            
        # If no extra features available, return standard loss
        return loss, loss_items

In [ ]:
%%writefile utils/dual_model.py
"""
Enhanced detection model for RGB+Range dual network implementation.
"""

import torch
import torch.nn as nn
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.loss import v8DetectionLoss

from utils.dual_loss import DualDetectionLoss


class DualDetectionModel(DetectionModel):
    """
    Enhanced YOLO detection model for dual RGB+Range networks.
    
    This class extends the standard DetectionModel to support:
    1. Feature extraction from both RGB and range/depth channels
    2. Custom loss functions specific to dual-modality inputs
    3. Intermediate feature storage for loss computation
    """
    
    def __init__(self, cfg="yolo8n.yaml", ch=4, nc=None, verbose=True):
        """Initialize dual detection model with 4 channel input by default."""
        super().__init__(cfg=cfg, ch=ch, nc=nc, verbose=verbose)
        
        # Create feature extraction layers
        self.rgb_extract = nn.Conv2d(3, 128, kernel_size=1)
        self.range_extract = nn.Conv2d(1, 128, kernel_size=1)
        
        # Initialize these on the same device as the model
        self.rgb_extract.to(self.device)
        self.range_extract.to(self.device)
        
        # Storage for intermediate features
        self.rgb_features = None
        self.range_features = None
        self.depth_map = None
        
    def _extract_modality_features(self, x):
        """Extract separate features for RGB and range channels."""
        # Split input into RGB and range components
        # x shape: [batch, 4, height, width]
        rgb = x[:, :3]  # First 3 channels (RGB)
        depth = x[:, 3:4]  # 4th channel (range/depth)
        
        # Save depth map for loss calculation
        self.depth_map = depth
        
        # Extract features
        self.rgb_features = self.rgb_extract(rgb)
        self.range_features = self.range_extract(depth)
        
        return x
    
    def _predict_once(self, x, profile=False, visualize=False, embed=None):
        """Modified forward pass with feature extraction."""
        # Extract RGB and range features
        x = self._extract_modality_features(x)
        
        # Continue with standard forward pass
        return super()._predict_once(x, profile, visualize, embed)
    
    def forward(self, x, *args, **kwargs):
        """Override forward to include RGB and range features in output."""
        if isinstance(x, dict):  # Training mode
            output = super().forward(x, *args, **kwargs)
            
            # For loss computation in training, pass features too
            if self.training:
                # If output is a tuple of (loss, loss_items)
                if isinstance(output, tuple) and len(output) == 2:
                    # Add a third element to the tuple containing our features
                    features_dict = {
                        'rgb_features': self.rgb_features,
                        'range_features': self.range_features,
                        'depth_map': self.depth_map
                    }
                    return (output[0], output[1], features_dict)
            return output
        else:  # Inference mode
            return super().forward(x, *args, **kwargs)
    
    def init_criterion(self):
        """Initialize the custom dual detection loss."""
        return DualDetectionLoss(self)


# Register the custom model class for loading/saving
torch.serialization.add_safe_globals([
    DualDetectionModel
])

In [ ]:
# Import our custom model
import sys
sys.path.append('.')  # Add current directory to path to import our modules

from utils.dual_model import DualDetectionModel
from ultralytics import YOLO

# Create a custom dual network model with our enhanced loss functions
dual_model = DualDetectionModel("yolov8n.pt", ch=4, nc=1)  # 4 channels, 1 class (snow poles)

# Train with Depth-Aware Loss
dual_model.train(
    data=str(DUAL_DATA_YAML),
    epochs=100,  # Reduced for faster experimentation
    imgsz=1024,
    device=0,
    amp=False,  # Disable AMP
    batch=16,
    project="dual_comb_range_experiments",
    name="depth_aware_loss",
    # Custom hyperparameters to prioritize depth-aware loss
    hyp={
        "depth": 1.0,    # Higher weight for depth-aware loss
        "modal": 0.3,    # Standard weight for modal loss
        "consist": 0.2   # Standard weight for consistency loss
    }
)

In [ ]:
# Train with Cross-Modal Attention Loss
dual_model = DualDetectionModel("yolov8n.pt", ch=4, nc=1)

dual_model.train(
    data=str(DUAL_DATA_YAML),
    epochs=100,
    imgsz=1024,
    device=0,
    amp=False,
    batch=16,
    project="dual_comb_range_experiments",
    name="cross_modal_attention_loss",
    # Custom hyperparameters to prioritize cross-modal attention loss
    hyp={
        "depth": 0.3,    # Standard weight for depth-aware loss
        "modal": 1.0,    # Higher weight for modal loss
        "consist": 0.2   # Standard weight for consistency loss
    }
)

In [ ]:
# Train with Feature Consistency Loss
dual_model = DualDetectionModel("yolov8n.pt", ch=4, nc=1)

dual_model.train(
    data=str(DUAL_DATA_YAML),
    epochs=100,
    imgsz=1024,
    device=0,
    amp=False,
    batch=16,
    project="dual_comb_range_experiments",
    name="feature_consistency_loss",
    # Custom hyperparameters to prioritize feature consistency loss
    hyp={
        "depth": 0.3,    # Standard weight for depth-aware loss
        "modal": 0.2,    # Standard weight for modal loss
        "consist": 1.0   # Higher weight for consistency loss
    }
)

In [ ]:
# Combined approach with balanced weights for all loss functions
dual_model = DualDetectionModel("yolov8n.pt", ch=4, nc=1)

dual_model.train(
    data=str(DUAL_DATA_YAML),
    epochs=100,
    imgsz=1024,
    device=0,
    amp=False,
    batch=16,
    project="dual_comb_range_experiments",
    name="combined_balanced_loss",
    # Balanced weights for all loss components
    hyp={
        "depth": 0.5,    # Balanced weight for depth-aware loss
        "modal": 0.5,    # Balanced weight for modal loss
        "consist": 0.5   # Balanced weight for consistency loss
    }
)

# Analysis of Custom Loss Functions for Dual Networks

## Theoretical Advantages of Each Loss Function

### 1. Depth-Aware Loss
This loss function is particularly beneficial for lidar-based snowpole detection because:
- **Depth-dependent difficulty**: Objects at different depths present different challenges. Far objects appear smaller and with less detail, while very close objects might be partially visible.
- **Range focus**: By weighting the loss based on range values, we can focus the model's attention on depth ranges where snowpoles are most critical to detect.
- **Handling difficult regions**: The Gaussian weighting reduces penalties in very challenging depth regions, allowing the model to focus on more reliable detections.

### 2. Cross-Modal Attention Loss
This approach should improve performance because:
- **Complementary information**: RGB provides color and texture while range provides precise depth and shape information. This loss ensures the model learns to leverage both.
- **Focused feature fusion**: By encouraging complementary attention between modalities, the model can focus on what each modality does best.
- **Balanced attention**: Using Jensen-Shannon divergence with a target similarity of 0.5 ensures the model doesn't overly rely on either modality.

### 3. Feature Consistency Loss
This method offers these advantages:
- **Aligned representations**: Forces the model to develop consistent internal representations of objects across modalities.
- **Robustness to sensor failures**: If representations are consistent, the model can better handle cases where one sensor provides degraded information.
- **Object-focused**: By masking to object regions, the consistency is focused where it matters most.

### Expected Best Performer

For snowpole detection in lidar data, I expect the **Depth-Aware Loss** to perform best as a single approach because:
- Range information is critically important for distinguishing snowpoles from other vertical objects
- Depth-based weighting directly addresses the varying difficulty of detection at different ranges
- It specifically focuses on the most relevant depth ranges for snowpole detection

However, the **Combined Balanced Approach** may yield the best overall results by leveraging the strengths of all three methods:
- Depth-aware loss handles range-specific challenges
- Cross-modal attention ensures optimal fusion of RGB and range features
- Feature consistency creates robust representations that work across modalities

## Monitoring and Validation

When evaluating these methods, look for:
1. **Higher mAP at different depth ranges**: especially improvement in challenging depth zones
2. **Reduced false positives**: particularly in ambiguous scenarios with similar vertical objects
3. **Generalization**: better performance in new environments or adverse weather conditions
4. **Faster convergence**: models may reach peak performance in fewer epochs
5. **Feature visualizations**: examine if the model focuses on the right areas in each modality

## Supporting Literature

- **Depth-aware weighting** – Zhang, S., Shih, K. J., Tai, Y.-W., & Tang, C.-K. “Deep Depth-Aware Features for RGB-D Object Detection.” *ECCV 2018*. Demonstrates that explicitly weighting detection signals with depth priors improves RGB-D object detection and motivates the Gaussian depth-weighting term we use.
- **Cross-modal attention** – Wei, L., Zhang, J., Ji, Y., Yan, C., & Fan, D.-P. “UCNet: Uncertainty Guided Cross-Modal Attention for RGB-D Salient Object Detection.” *CVPR 2020*. Shows that cross-modal attention mechanisms let RGB and depth streams emphasize complementary cues, underpinning our cross-modal attention loss.
- **Feature consistency across modalities** – Chen, X., Li, L., & Li, S. “Cross-Modal Consistency Learning for RGB-D Semantic Segmentation.” *ICCV 2021*. Introduces a consistency regularizer between RGB and depth feature spaces to improve robustness, which inspired our cosine-similarity feature consistency loss.

In [ ]:
from ultralytics import YOLO

# if still in memory
# model = model

# or reload from best checkpoint
# model = YOLO("path/to/runs/detect/dual_comb_rgb_plus_range_v8n_4ch/weights/best.pt")

metrics = model.val(
    data=str(DUAL_DATA_YAML),  # your data.yaml with test path
    split="test",              # use test set instead of val
    imgsz=1024,
    device=0,                # GPU id, or "cpu"
    batch=16,
    project="dual_comb_range_experiments",
    name="dual_comb_rgb_plus_range_v8n_4ch_test",
)

print(metrics)  # mAP50, mAP50-95, precision, recall, etc.


Ultralytics YOLOv8.2.5 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
Model summary (fused): 168 layers, 3005987 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/labels/test... 197 images, 0 backgrounds, 0 corrupt: 100%|██████████| 197/197 [00:51<00:00,  3.81it/s]

val: New cache created: /content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:02<00:00,  6.29it/s]


                   all        197        395      0.857      0.834      0.883      0.449
Speed: 0.1ms preprocess, 1.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to dual_comb_range_experiments/dual_comb_rgb_plus_range_v8n_4ch_test
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ce66060ff80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029, 